# CIFAR-10 CNN: 1層のConv2dをTucker-2で圧縮する

このNotebookでは、`00_fundamentals` で確認したHOSVD/Tuckerを **CNNのConv2d重みへ接続する**。

最初からモデル全体は圧縮しない。まずCIFAR-10 baselineのConv層を1つだけ選び、

\[
W \in \mathbb{R}^{C_{out}\times C_{in}\times k_H\times k_W}
\]

をmode 0 (`out channel`) と mode 1 (`in channel`) だけでTucker分解する。

このNotebookのゴールは、次の対応を自分で実装して説明できること。

```text
元のConv2d
    ↓ Tucker-2
1x1 Conv: Cin -> rank_in
    ↓
kHxkW Conv: rank_in -> rank_out
    ↓
1x1 Conv: rank_out -> Cout
```

ここでは **1層の置換まで** を扱う。rank sweep・fine-tuning・複数層圧縮は後のNotebookで行う。


## 1. baseline CNNとTucker共通処理を準備する

既存の `CIFAR10CNN` と、`00_fundamentals` から `src` に共通化したTucker処理を使う。

Cursor側で共通化した後、実際のexport先に合わせてimportを確認すること。

baseline checkpointは既存の

`models/10_svd/40_cifar10_cnn/01_cnn_cifar10_baseline/cifar10_cnn_baseline.pt`

を再利用する。


In [ ]:
from pathlib import Path
import copy

import torch
from torch import nn

from nn_compression.models.cifar10 import CIFAR10CNN

# TODO:
# CursorでTucker処理をsrcへ共通化した後、実際のexport先からimportする。
# 例:
# from nn_compression.compression import hosvd, reconstruct_tucker, tucker_parameter_count
# from nn_compression.metrics import relative_frobenius_error

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "nn-compression-svd-dmrg" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_PATH = (
    PROJECT_ROOT
    / "models/10_svd/40_cifar10_cnn/01_cnn_cifar10_baseline"
    / "cifar10_cnn_baseline.pt"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 2. baselineモデルを読み込み、圧縮するConv層を1つ選ぶ

最初は `conv2` を対象にする。

`conv2.weight` のshapeを確認して、各軸が何を表しているか書けるようにする。

```text
mode 0 = out_channels
mode 1 = in_channels
mode 2 = kH
mode 3 = kW
```

このNotebookでは `groups=1` の通常Convだけを扱う。


In [ ]:
model = CIFAR10CNN().to(device)

# TODO:
# checkpointの保存形式を確認してbaselineの重みを読み込む。
# state_dictそのものか、dict内の"model_state_dict"等かを確認して対応する。

target_conv = model.conv2

print("weight shape:", tuple(target_conv.weight.shape))
print("bias:", target_conv.bias is not None)
print("stride:", target_conv.stride)
print("padding:", target_conv.padding)
print("groups:", target_conv.groups)


## 3. Conv2d重みをTucker-2分解する

Tucker-2では空間方向 `kH`, `kW` は圧縮せず、channel方向だけを圧縮する。

したがって `hosvd` へ渡すrankは

```python
{0: rank_out, 1: rank_in}
```

になる。

分解後に確認するshape:

```text
U_out : (Cout, rank_out)
U_in  : (Cin, rank_in)
core  : (rank_out, rank_in, kH, kW)
```

まずは適当な `rank_out`, `rank_in` を1組だけ選び、shapeが上の対応になることを確認する。


In [ ]:
rank_out = 32
rank_in = 16

def tucker2_decompose_conv_weight(
    weight: torch.Tensor,
    rank_out: int,
    rank_in: int,
):
    """
    目的:
        Conv2d weightをmode 0 / mode 1だけHOSVDし、
        core, U_out, U_inを返す。

    weight:
        (Cout, Cin, kH, kW)

    返り値:
        core
        U_out
        U_in
    """
    # HINT:
    # ranks = {0: rank_out, 1: rank_in}
    # 00_fundamentalsで作ったhosvdを使う。
    pass


## 4. Tucker-2の3つの因子をConv2dへ対応させる

Tucker-2のテンソル表現を、実際に計算できる3層のConvへ変換する。

shape対応は次の通り。

```text
U_in.T
(Cin, rank_in)
    ↓ reshape
1x1 Conv weight
(rank_in, Cin, 1, 1)

core
(rank_out, rank_in, kH, kW)
    ↓
中央のkHxkW Conv weight

U_out
(Cout, rank_out)
    ↓ reshape
最後の1x1 Conv weight
(Cout, rank_out, 1, 1)
```

元Convの空間方向の `stride / padding / dilation` は中央のConvが担当する。

biasがある場合は、元Convと同じ出力channelに対応する **最後の1x1 Conv** に持たせる。

このNotebookではまず `groups=1` のみ対応する。


In [ ]:
def build_tucker2_conv(
    conv: nn.Conv2d,
    rank_out: int,
    rank_in: int,
) -> nn.Sequential:
    """
    目的:
        1つのConv2dをTucker-2分解し、
        1x1 -> kHxkW -> 1x1 のnn.Sequentialへ置き換える。

    conv:
        groups=1のnn.Conv2d

    rank_out:
        out channel側のTucker rank

    rank_in:
        in channel側のTucker rank
    """
    # TODO:
    # 1. conv.weightをTucker-2分解
    # 2. 3つのConv2dを作る
    # 3. U_in / core / U_outを各weightへ対応させる
    # 4. 元biasがあれば最後の1x1 Convへコピーする
    pass


## 5. 1層だけで出力shapeと近似誤差を確認する

いきなりCNN全体へ組み込まず、まず `target_conv` とTucker-2 Convへ同じ入力を入れる。

確認するもの:

- 出力shapeが一致する
- rankを落としているので出力値は完全一致しなくてよい
- weight再構成誤差と出力誤差を区別する

入力Tensorは `target_conv.in_channels` に合わせて作る。


In [ ]:
# TODO:
# tucker_conv = build_tucker2_conv(target_conv, rank_out, rank_in).to(device)
#
# x = torch.randn(
#     4,
#     target_conv.in_channels,
#     16,
#     16,
#     device=device,
# )
#
# with torch.no_grad():
#     y_original = target_conv(x)
#     y_tucker = tucker_conv(x)
#
# print(y_original.shape, y_tucker.shape)
# 出力の相対誤差も計算する。


## 6. パラメータ数を比較する

元Convのweightパラメータ数は

```text
Cout * Cin * kH * kW
```

Tucker-2後は

```text
Cin * rank_in
+ rank_out * rank_in * kH * kW
+ Cout * rank_out
```

になる。

biasを数える場合は元Conv/Tucker-2の両方で同じ条件に揃える。

`00_fundamentals` の `tucker_parameter_count` で数えた値と、
実際に作った3層Convのweight要素数が一致することも確認する。


In [ ]:
# TODO:
# original_params = ...
# tucker_params_formula = ...
# tucker_params_module = ...
#
# print(...)


## 7. baseline CNNのconv2だけ置き換える

`copy.deepcopy(model)` でbaselineを残し、コピー側の `conv2` だけをTucker-2 Convへ置き換える。

ここでは **モデル全体を一括圧縮しない**。

まず以下を確認する。

- forwardが通る
- logits shapeが `(batch_size, 10)` のまま
- baselineモデル自体は変更されていない

精度評価は次のrank sweep Notebookで本格的に行う。


In [ ]:
compressed_model = copy.deepcopy(model)

# TODO:
# compressed_model.conv2 = ...
# dummy = torch.randn(4, 3, 32, 32, device=device)
# with torch.no_grad():
#     logits = compressed_model(dummy)
# print(logits.shape)


## 8. TensorLyで検算する

最後にTensorLyの `partial_tucker` を使って、考え方が大きくずれていないか検算する。

注意:

- 自作 `hosvd` はone-pass HOSVD
- TensorLyの `partial_tucker` は反復改善を行う
- factor matrixの値そのものの一致は要求しない

比較するもの:

- core / factorのshape
- 再構成shape
- relative Frobenius error

TensorLyを「答えの実装」として先に使わず、自作実装ができた後の検算に使う。


In [ ]:
# TODO:
# from tensorly.decomposition import partial_tucker
# TensorLyでmode 0,1だけ分解し、shapeと再構成誤差を比較する。


## 9. このNotebookの完了条件

次を説明・確認できれば `02_rank_sweep.ipynb` へ進む。

1. Conv2d weightの4軸が何を表すか
2. Tucker-2でなぜmode 0 / 1だけを圧縮するか
3. `U_in -> core -> U_out` がなぜ `1x1 -> kxk -> 1x1` になるか
4. 元Convのbiasを最後の1x1 Convへ置く理由
5. 元ConvとTucker-2 Convのパラメータ数を計算できる
6. CNNのConv層を1つだけ実際に置換してforwardできる
